# HW5 · Monitoring（OpenTelemetry）

> 官方：[homework.md](../llm-zoomcamp/cohorts/2026/05-monitoring/homework.md) · 中文：[homework.zh.md](../llm-zoomcamp/cohorts/2026/05-monitoring/homework.zh.md)  
> 提交：https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw5  
> Deadline：7/20 23:00

**心智模型**

```
一次 rag(query) = 一条 Trace
  └─ span "rag"
       ├─ span "search"   ← 本地 minsearch（快）
       └─ span "llm"      ← OpenAI（慢；挂 tokens / cost）
```

| 题 | 要点 | 答案 |
|----|------|------|
| Q1 | span 数量 |  |
| Q2 | input_tokens |  |
| Q3 | LLM 耗时区间 |  |
| Q4 | spans 表里的 span 名 |  |
| Q5 | 总耗时最长（排除 rag） |  |
| Q6 | 4 次 input_tokens 稳定性 |  |

## Phase 0 · 环境

确保项目根目录有 `.env`（`OPENAI_API_KEY=...`），并用 `uv sync` / 本 notebook 的 kernel（`.venv`）。

In [1]:
from dotenv import load_dotenv
load_dotenv()

QUERY = "How does the agentic loop keep calling the model until it stops?"
QUERY

'How does the agentic loop keep calling the model until it stops?'

## Phase 1 · OTel + `RAGTraced` → Q1

**先设 TracerProvider，再 import starter。**  
若重复跑本 cell，可能看到 TracerProvider 已被设置的警告，可 Restart Kernel 后从这里重来。

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

print("tracer ready:", tracer)

tracer ready: <opentelemetry.sdk.trace.Tracer object at 0x10a7e8d40>


In [3]:
# 首次会从 GitHub 拉 72 篇 lessons，稍慢
from starter import index, client
from rag_helper import RAGBase


class RAGTraced(RAGBase):
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            return super().llm(prompt)

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)


rag = RAGTraced(index=index, llm_client=client)
print("model:", rag.model)

model: gpt-5.4-mini


In [4]:
# Q1：看控制台打印了几个 ReadableSpan（字典块）
answer = rag.rag(QUERY)
print(answer[:500])

{
    "name": "search",
    "context": {
        "trace_id": "0x0afe1b94e359edc026c4536732ff65be",
        "span_id": "0x0f2e4d9bbbb4ce09",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8b0cae810c75fba0",
    "start_time": "2026-07-14T15:57:01.163431Z",
    "end_time": "2026-07-14T15:57:01.168815Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "3a86e1ef-abc0-44fc-abfa-876ed28290aa",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x0afe1b94e359edc026c4536732ff65be",
        "span_id": "0x5d20dfe1b8329618",
        "trace_state": "[]"
    },
    "kind": "SpanKind

### Q1 记录

控制台里有几个 finished span？

3

## Phase 2 · Attributes（tokens + cost）→ Q2 / Q3

给 `llm` span 挂上 `input_tokens` / `output_tokens` / `cost`，再跑一次查询。

In [5]:
def calculate_cost(model, usage):
    if "gpt-5.4-mini" in model:
        return (usage.input_tokens * 0.15 + usage.output_tokens * 0.60) / 1_000_000
    return 0.0


class RAGTraced(RAGBase):
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            span.set_attribute("cost", calculate_cost(self.model, usage))
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)


rag = RAGTraced(index=index, llm_client=client)
answer = rag.rag(QUERY)
print(answer[:300])

{
    "name": "search",
    "context": {
        "trace_id": "0xece21bf584b3651d974ed9e69b0c3385",
        "span_id": "0x1eff9a9e1d2e9d42",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1cb99d25c89e15c2",
    "start_time": "2026-07-14T15:58:23.513746Z",
    "end_time": "2026-07-14T15:58:23.516384Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "3a86e1ef-abc0-44fc-abfa-876ed28290aa",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xece21bf584b3651d974ed9e69b0c3385",
        "span_id": "0x7a38f220267ddec8",
        "trace_state": "[]"
    },
    "kind": "SpanKind

### Q2 / Q3 记录

在 console 输出的 `llm` span 里找 `input_tokens`，在 `search` / `llm` 上算耗时：

`duration_ms = (end_time - start_time) / 1e6`

**Q2 input_tokens（选最接近）：**

7111

**我的答案 / 实际值：** 

**Q3 LLM 常见耗时：**

1888

## Phase 3 · SQLite exporter → Q4

换掉 Console，把 span 写入 `traces.db`。  
想干净样本可先删旧库：下一格会重建表。

In [6]:
import os
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

DB_PATH = "traces.db"

# 想干净样本时取消注释：
# if os.path.exists(DB_PATH):
#     os.remove(DB_PATH)


class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path=DB_PATH):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute(
            """
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
            """
        )
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True


# 推荐：Restart Kernel 后，跳过 Console 那格，只跑本格（只用 SQLite）
# 若未重启、provider 已存在：只能追加 processor（console + sqlite 会同时写）
current = trace.get_tracer_provider()
if type(current).__name__ == "TracerProvider":
    current.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(DB_PATH)))
    provider = current
    print("appended SQLite exporter ->", DB_PATH)
else:
    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(DB_PATH)))
    trace.set_tracer_provider(provider)
    print("new TracerProvider with SQLite ->", DB_PATH)

tracer = trace.get_tracer("llm-zoomcamp")

appended SQLite exporter -> traces.db


In [7]:
answer = rag.rag(QUERY)
print(answer[:200])

conn = sqlite3.connect(DB_PATH)
print("distinct names:", conn.execute("SELECT DISTINCT name FROM spans").fetchall())
print("rows:", conn.execute("SELECT name, input_tokens, output_tokens, cost FROM spans").fetchall())
conn.close()

{
    "name": "search",
    "context": {
        "trace_id": "0x8f31e30f7e781e4a1fc863a902d224cf",
        "span_id": "0x10626d3007664d41",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x7bd07f93d7f0fa8b",
    "start_time": "2026-07-14T16:00:30.177387Z",
    "end_time": "2026-07-14T16:00:30.179419Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "3a86e1ef-abc0-44fc-abfa-876ed28290aa",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x8f31e30f7e781e4a1fc863a902d224cf",
        "span_id": "0x12d2dcbfb31d6256",
        "trace_state": "[]"
    },
    "kind": "SpanKind

### Q4 记录

`spans` 表里出现哪些 span 名？

`rag`, `search`, and `llm`

## Phase 4 · 排除 `rag`，比总耗时 → Q5

再跑至少一次，增加样本；父 span `rag` 要排除，否则总会最长。

In [8]:
answer = rag.rag(QUERY)
print(answer[:150])

{
    "name": "search",
    "context": {
        "trace_id": "0x69aeb7f5968daa0b0c92767c1f4e95a1",
        "span_id": "0x51e6b629253772cc",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x61e727f4d11fecec",
    "start_time": "2026-07-14T16:01:25.889236Z",
    "end_time": "2026-07-14T16:01:25.891652Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "3a86e1ef-abc0-44fc-abfa-876ed28290aa",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x69aeb7f5968daa0b0c92767c1f4e95a1",
        "span_id": "0x49287301d2a92342",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [9]:
import pandas as pd

df = pd.read_sql("SELECT * FROM spans", sqlite3.connect(DB_PATH))
df["duration_ns"] = df["end_time"] - df["start_time"]
df["duration_ms"] = df["duration_ns"] / 1e6

by_name = (
    df[df["name"] != "rag"]
    .groupby("name")["duration_ns"]
    .sum()
    .sort_values(ascending=False)
)
display(df)
display(by_name)
display((by_name / 1e6).rename("total_ms"))

,name,start_time,end_time,input_tokens,output_tokens,cost,duration_ns,duration_ms
0,search,1784044830177387000,1784044830179419000,NaN,NaN,NaN,2032000,2.032
1,llm,1784044830180816000,1784044831813464000,7111.0,118.0,0.001137,1632648000,1632.648
2,rag,1784044830177353000,1784044831817940000,NaN,NaN,NaN,1640587000,1640.587
3,search,1784044885889236000,1784044885891652000,NaN,NaN,NaN,2416000,2.416
4,llm,1784044885893295000,1784044888746675000,7111.0,125.0,0.001142,2853380000,2853.380
5,rag,1784044885889198000,1784044888750874000,NaN,NaN,NaN,2861676000,2861.676


name
llm       4486028000
search       4448000
Name: duration_ns, dtype: int64

name
llm       4486.028
search       4.448
Name: total_ms, dtype: float64

### Q5 记录

排除 `rag` 后，哪类 span 总耗时最长？

- [X] `llm`

## Phase 5 · 同一 query 跑满 4 次 → Q6

库里要有 **4 条**带 `input_tokens` 的 `llm` span。按已有次数调整下面的 `range`。

In [10]:
conn = sqlite3.connect(DB_PATH)
n_llm = conn.execute(
    "SELECT COUNT(*) FROM spans WHERE name = 'llm' AND input_tokens IS NOT NULL"
).fetchone()[0]
print("current llm rows with tokens:", n_llm)

need = max(0, 4 - n_llm)
for i in range(need):
    print(f"run {i + 1}/{need}...")
    rag.rag(QUERY)

tokens = pd.read_sql(
    "SELECT input_tokens FROM spans WHERE name = 'llm' AND input_tokens IS NOT NULL",
    sqlite3.connect(DB_PATH),
)
display(tokens)
display(tokens["input_tokens"].describe())
print("unique values:", sorted(tokens["input_tokens"].unique().tolist()))

current llm rows with tokens: 2
run 1/2...
{
    "name": "search",
    "context": {
        "trace_id": "0xe2e931c391ff01325588f50be39a0462",
        "span_id": "0x4d194cc7eae0b1fc",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x4a04bc51885e2828",
    "start_time": "2026-07-14T16:03:07.561779Z",
    "end_time": "2026-07-14T16:03:07.563523Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "3a86e1ef-abc0-44fc-abfa-876ed28290aa",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xe2e931c391ff01325588f50be39a0462",
        "span_id": "0xc5714d0bedff0e06",
        "trac

,input_tokens
0,7111
1,7111
2,7111
3,7111


count       4.0
mean     7111.0
std         0.0
min      7111.0
25%      7111.0
50%      7111.0
75%      7111.0
max      7111.0
Name: input_tokens, dtype: float64

unique values: [7111]


### Q6 记录

4 次 `input_tokens` 波动？

- [X] They're identical